# Etude No. 1: Multi-Laminar Cortical AGSDR Scaffold

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/etudes/jaxfne_etude_no_1_multi_laminar_cortical_agsdr.ipynb)

**End-to-end jaxfne workflow:** single-unit warmup, configure, build, simulate, optimize, visualize, export.

## Scope Gates
- **truth_mode:** `truth_safe_unverified`
- **claim_level:** `computational_scaffold`
- **field_solver_status:** `laminar_proxy_no_pde`
- **physical_amplitude_claim_allowed:** `False`

All outputs are simulated/proxy diagnostics. No physical-amplitude, biological-calibration, or mechanism claim is made.

## Colab Installation: PyPI Release

In [ ]:
!pip install -q "jaxfne[viz]"

## Colab Installation: Release Branch (main)

Run this cell to install the current `main` release candidate before PyPI catches up.

In [ ]:
# Colab/runtime guard: install current GitHub main with visualization extras.
# Force-reinstall ensures a stale cached wheel does not silently shadow the patch.
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade", "--force-reinstall", "--no-cache-dir",
    "jaxfne[viz] @ git+https://github.com/HNXJ/jaxfne.git@main",
])

## Imports

In [ ]:
import os, json, hashlib, pandas as pd; from pathlib import Path
import matplotlib.pyplot as plt; import numpy as np; import jax
import jaxfne as jtfne
def _maybe_show_plotly(fig):
    """Write Plotly HTML artifacts without blocking headless nbclient runs."""
    if os.environ.get("TFNE_RENDER_PLOTLY", "0") == "1":
        fig.show(renderer=os.environ.get("PLOTLY_RENDERER", "notebook_connected"))
    return fig

def _maybe_show_matplotlib(fig):
    """Render matplotlib figures only when explicitly requested."""
    if os.environ.get("TFNE_RENDER_MPL", "0") == "1":
        try:
            from IPython.display import display
            display(fig)
        except Exception:
            _maybe_show_matplotlib(fig)
    return fig


## Install Verification

Confirm the patched package is active before proceeding.
If `visualize_network_3d` is missing, use **Runtime → Restart runtime** and re-run from the top.

In [ ]:
# Verify the installed package is the patched version from GitHub main.
import importlib.metadata as _md

print("jaxfne version:", _md.version("jaxfne"))
print("jaxfne path:   ", jtfne.__file__)
print("vis path:      ", jtfne.vis.__file__)

_required = "visualize_network_3d"
if not hasattr(jtfne.vis, _required):
    raise RuntimeError(
        f"Installed jaxfne.vis is missing {_required}. "
        "Colab is likely using a stale package or runtime cache. "
        "Use Runtime → Restart runtime, then re-run from the top.\n"
        f"jaxfne path: {jtfne.__file__}; "
        f"vis path: {getattr(jtfne.vis, '__file__', None)}"
    )
print("OK visualize_network_3d present in jtfne.vis")

## Centralized Full-Detail Config

All knobs in one place. Every subsequent cell derives its values from `config`.
This cell may exceed the normal code-cell line preference — it is the main edit anchor.

In [ ]:
from types import SimpleNamespace
from pathlib import Path as _Path

config = SimpleNamespace(
    # === Runtime ===
    SEED          = 20260530,
    DT_MS         = 0.1,
    T_MS_DEFAULT  = 1000.0,
    SMOKE_T_MS    = 300.0,
    DTYPE         = "float32",
    # === Geometry (declarative; proxy scaffold only) ===
    CX_M          = 1.0e-3,
    CY_M          = 1.0e-3,
    CZ_M          = 1.6e-3,
    UM_PER_M      = 1.0e6,
    COLUMN_RADIUS_MM = 0.25,
    # === Areas & Columns ===
    AREA_ORDER    = ["V1", "V4"],
    N_PER_AREA    = 80,
    N_PER_AREA_SMOKE = 40,
    # === Layers ===
    LAYERS        = ["L1", "L2/3", "L4", "L5", "L6"],
    LAYER_FRACTIONS = {
        "L1":  (0.00, 0.10),
        "L2/3":(0.10, 0.30),
        "L4":  (0.30, 0.45),
        "L5":  (0.45, 0.70),
        "L6":  (0.70, 1.00),
    },
    # === Cell Types ===
    CELL_TYPES    = ["E", "PV", "SST", "VIP"],
    CELL_COLORS   = {"E": "#e69500", "PV": "#0072ce", "SST": "#ffbf00", "VIP": "#7b3294"},
    CELL_SIGNS    = {"E": 1.0, "PV": -1.0, "SST": -1.0, "VIP": -1.0},
    CELL_TYPE_FRACTIONS = {"E": 0.75, "PV": 0.10, "SST": 0.08, "VIP": 0.07},
    LAYER_CELL_FRACTIONS = {
        "L1":   {"E": 0.85, "PV": 0.00, "SST": 0.00, "VIP": 0.15},
        "L2/3": {"E": 0.78, "PV": 0.08, "SST": 0.07, "VIP": 0.07},
        "L4":   {"E": 0.60, "PV": 0.25, "SST": 0.10, "VIP": 0.05},
        "L5":   {"E": 0.70, "PV": 0.12, "SST": 0.12, "VIP": 0.06},
        "L6":   {"E": 0.72, "PV": 0.10, "SST": 0.08, "VIP": 0.10},
    },
    # === Drive & Noise (proxy metadata; not calibrated physical drive) ===
    DRIVE_BASELINE = {"E": 5.0, "PV": 3.0, "SST": 3.5, "VIP": 3.0},
    DRIVE_RANGE    = {
        "E":   (4.5, 9.0),
        "PV":  (2.5, 7.0),
        "SST": (2.5, 6.5),
        "VIP": (2.5, 6.5),
    },
    NOISE_SCALE    = 0.5,
    # === Inter-column connectivity (declarative metadata) ===
    INTER_P_FF     = 0.3,
    INTER_P_FB     = 0.2,
    INTER_FF_W     = (0.5, 2.0),
    INTER_FB_W     = (0.3, 1.5),
    # === Field / Probe / Readout ===
    N_CONTACTS     = 16,
    FIELD_SOLVER   = "laminar_proxy_no_pde",
    FREQ_MIN_HZ    = 1.0,
    FREQ_MAX_HZ    = 150.0,
    FREQ_COUNT     = 128,
    PSD_NPERSEG    = 128,
    SPECTRO_FIGSIZE = (14, 5),
    SPECTRO_CMAP   = "viridis",
    ALPHA_BETA_HZ  = (8.0, 25.0),
    GAMMA_HZ       = (40.0, 150.0),
    # === AGSDR Optimizer ===
    AGSDR_ALPHA    = 0.7,
    AGSDR_EXPL     = 0.05,
    AGSDR_DESEL    = 2.0,
    AGSDR_GEN      = 25,       # full run: 25 generations
    AGSDR_GEN_SMOKE = 3,       # smoke: 3 generations
    AGSDR_POP      = 6,        # full run population
    AGSDR_POP_SMOKE = 2,       # smoke population
    AGSDR_PARAM    = {"drive_gain": (0.1, 1.5)},
    # === Objective ===
    TARGET_RATE_HZ = 3.5,
    TARGET_KAPPA   = 0.0,
    RATE_WEIGHT    = 1.0,
    KAPPA_WEIGHT   = 0.25,
    # === Stimulus ===
    STIM_AREA      = "V1",
    STIM_LAYER     = "L4",
    STIM_CTYPE     = "E",
    STIM_ONSET_MS  = 100.0,
    STIM_DUR_MS    = 100.0,
    STIM_AMP       = 1.5,
    # === Visualization ===
    FIG_DPI        = 150,
    ACT_FIGSIZE    = (14, 8),
    CIRC_FIGSIZE   = (12, 5),
    # === Output paths ===
    OUTPUT_DIR     = _Path("outputs/etude_no_1"),
    # === Truth gates ===
    TRUTH_MODE     = "truth_safe_unverified",
    CLAIM_LEVEL    = "computational_scaffold",
    FIELD_SOLVER_STATUS = "laminar_proxy_no_pde",
    PHYSICAL_AMPLITUDE_CLAIM_ALLOWED = False,
)

## Finalize Config (derived paths)

Creates output directories and attaches derived `FIG_DIR` and `PLOTLY_DIR`.

In [ ]:
def finalize_config(cfg):
    cfg.FIG_DIR    = cfg.OUTPUT_DIR / "figures"
    cfg.PLOTLY_DIR = cfg.OUTPUT_DIR / "plotly"
    cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cfg.FIG_DIR.mkdir(parents=True, exist_ok=True)
    cfg.PLOTLY_DIR.mkdir(parents=True, exist_ok=True)
    return cfg

config = finalize_config(config)

## Runtime Constants (derived from config)

`TFNE_SMOKE=1` selects a fast smoke configuration. All values come from the
centralized `config` object above.

In [ ]:
SMOKE       = os.environ.get("TFNE_SMOKE", "0") == "1"
SEED        = config.SEED
DT_MS       = config.DT_MS
DTYPE       = config.DTYPE
DURATION_MS = config.SMOKE_T_MS if SMOKE else config.T_MS_DEFAULT
N_PER_AREA  = config.N_PER_AREA_SMOKE if SMOKE else config.N_PER_AREA
AREAS       = config.AREA_ORDER
OUTPUT_DIR  = config.OUTPUT_DIR
FIG_DIR     = config.FIG_DIR

# Part 1 — Single-Unit Izhikevich Warmup

Before building the multi-area laminar scaffold, simulate one reduced native Izhikevich
unit for each cell class used later (E, PV, SST, VIP). The goal is to inspect native reduced
dynamics in isolation. This uses package-native engine functions only — no local solver.

## Warmup Configuration (editable)

In [ ]:
WARMUP_LABELS = ('E', 'PV', 'SST', 'VIP')
WARMUP_LAYERS = ('warmup', 'warmup', 'warmup', 'warmup')
WARMUP_DRIVE = {'E': 5.0, 'PV': 3.0, 'SST': 3.5, 'VIP': 3.0}
WARMUP_DURATION_MS = 300.0

## Build Native Reduced Izhikevich Params

All keyword arguments shown explicitly, including defaults.

In [ ]:
warmup_params = jtfne.izhikevich_params_from_labels(
    labels=WARMUP_LABELS, layer_labels=WARMUP_LAYERS,
    dtype=DTYPE, drive_overrides=WARMUP_DRIVE, source_scale=1.0,
)

## PRNG Key and Step Count

Explicit PRNG key (no implicit global RNG).

In [ ]:
warmup_key = jax.random.PRNGKey(SEED)
warmup_steps = int(round(WARMUP_DURATION_MS / DT_MS))

## Simulate Native Units

Returns `(V, spikes, sources)`; all keyword arguments shown explicitly.

In [ ]:
warmup_V, warmup_spikes, warmup_sources = jtfne.simulate_eig_izhikevich(
    params=warmup_params, n_steps=warmup_steps, dt_ms=DT_MS, key=warmup_key,
    dtype=DTYPE, drive_schedule=None, silence_mask=None,
)

## Warmup Firing Rates

In [ ]:
warmup_rate_hz = 1000.0 * np.asarray(warmup_spikes).sum(axis=0) / (warmup_steps * DT_MS)
warmup_rate_table = pd.DataFrame({'cell_type': list(WARMUP_LABELS), 'rate_hz': warmup_rate_hz})
print(warmup_rate_table.to_string(index=False))

## Warmup Voltage Traces

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
for i, label in enumerate(WARMUP_LABELS): ax.plot(np.asarray(warmup_V)[:, i], label=label)
ax.set(title='Single-unit Izhikevich warmup (native V_m)', xlabel='Step', ylabel='Native V_m')
ax.legend(); plt.close(fig)

# Part 2 — Multi-Laminar Cortical Scaffold

Configure a V1/V4 multi-laminar scaffold, simulate baseline and stimulus conditions,
tune with AGSDR toward a firing-rate and synchrony target, visualize, and export artifacts.
Every editable input is declared as a named variable group.

## Runtime & Column Domains (editable via config)

These dicts expose jaxfne-API-facing settings. Edit the centralized `config` cell
above to change values — the dicts here derive from it.

In [ ]:
runtime = {"seed": SEED, "duration_ms": DURATION_MS, "dt_ms": DT_MS, "dtype": DTYPE}
LAYERS  = config.LAYERS
columns = {"areas": AREAS, "n_per_area": N_PER_AREA, "layers": LAYERS}
cell_types = config.CELL_TYPE_FRACTIONS

## Drive, Inter-Column, Field (derived from config)

In [ ]:
drive     = {"baseline": config.DRIVE_BASELINE, "noise": config.NOISE_SCALE}
inter_conn = {"source": AREAS[0], "target": AREAS[-1],
              "p_ff": config.INTER_P_FF, "p_fb": config.INTER_P_FB}
field     = {"solver": config.FIELD_SOLVER, "domain": "laminar_column"}

## Probes, Objective, Optimizer (derived from config)

In [ ]:
probes    = {"types": ["spikes", "V_m", "source", "LFP", "CSD"],
             "n_contacts": config.N_CONTACTS}
objective = {"rate_hz": config.TARGET_RATE_HZ, "kappa": config.TARGET_KAPPA,
             "rate_w": config.RATE_WEIGHT, "kappa_w": config.KAPPA_WEIGHT}
_agsdr_gen = config.AGSDR_GEN_SMOKE if SMOKE else config.AGSDR_GEN
_agsdr_pop = config.AGSDR_POP_SMOKE if SMOKE else config.AGSDR_POP
optimizer = {"family": "AGSDR", "alpha": config.AGSDR_ALPHA,
             "exploration": config.AGSDR_EXPL, "deselect_factor": config.AGSDR_DESEL,
             "parameters": config.AGSDR_PARAM,
             "gen": _agsdr_gen, "pop": _agsdr_pop, "seed": SEED}

## Configuration Construction Arguments (editable)

All `default_spectrolaminar_config` keyword arguments declared explicitly.

In [ ]:
CFG_KWARGS = {'areas': AREAS, 'n_per_area': N_PER_AREA, 'seed': SEED,
              'duration_ms': DURATION_MS, 'dt_ms': DT_MS}

## Construct Configuration and Model

In [ ]:
cfg = jtfne.default_spectrolaminar_config(**CFG_KWARGS)
model = jtfne.construct(cfg); n_units = model.summary()['n_units']

## Visualization Helpers

`visualize_circuit` renders the 3D proxy geometry; `plot_activity_suite` shows
raster, firing-rate trace, LFP proxy, and CSD proxy for any `signals` object.
Both helpers display the figure in the notebook **and** save a PNG artifact.
These are setup cells and may exceed the normal workflow-cell line preference.

In [ ]:
def visualize_circuit(mdl, cfg):
    import pandas as pd
    neurons = pd.DataFrame(mdl.neuron_table())
    fig = plt.figure(figsize=cfg.CIRC_FIGSIZE, dpi=cfg.FIG_DPI)
    ax  = fig.add_subplot(111, projection="3d")
    for ct in cfg.CELL_TYPES:
        sub = neurons[neurons["cell_type"] == ct]
        ax.scatter(sub["x"], sub["y"], sub["z"], s=8,
                   c=cfg.CELL_COLORS[ct], label=ct, alpha=0.7, edgecolors="none")
    ax.set_xlabel("x (a.u.)"); ax.set_ylabel("y (a.u.)"); ax.set_zlabel("depth (a.u.)")
    ax.set_title("Cortical circuit geometry — proxy scaffold (laminar_proxy_no_pde)")
    ax.legend(loc="upper left", fontsize=8, markerscale=2)
    out = cfg.FIG_DIR / "cortical_circuit_network.png"
    fig.savefig(out, dpi=cfg.FIG_DPI, bbox_inches="tight")
    _maybe_show_matplotlib(fig)
    return fig

## Activity Suite Helper

In [ ]:
def plot_activity_suite(signals, label, cfg, fname_suffix=None):
    spikes = np.asarray(signals.spikes)
    t_ms   = np.asarray(signals.time_ms)
    lfp    = np.asarray(signals.field.lfp_proxy)
    csd    = np.asarray(signals.field.csd_proxy)
    depths = np.asarray(signals.field.contact_depths)
    n_steps, n_units = spikes.shape
    rate_hz = 1000.0 * spikes.mean(axis=1) / cfg.DT_MS
    fig, axes = plt.subplots(2, 2, figsize=cfg.ACT_FIGSIZE, dpi=cfg.FIG_DPI)
    fig.suptitle(f"Activity Suite — {label} (proxy · truth_safe_unverified)", fontsize=11)
    ax = axes[0, 0]; si, ti = np.where(spikes.T)
    ax.scatter(t_ms[ti], si, s=0.5, c="k", alpha=0.4)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Neuron"); ax.set_title("Spike raster")
    ax = axes[0, 1]
    ax.plot(t_ms, rate_hz, lw=1, color="#e69500")
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Rate (Hz)"); ax.set_title("Mean firing rate")
    ax = axes[1, 0]; vmax = float(np.abs(lfp).max()) or 1.0
    ax.imshow(lfp.T, aspect="auto", origin="lower",
              extent=[t_ms[0], t_ms[-1], float(depths[0]), float(depths[-1])],
              cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Depth"); ax.set_title("LFP proxy")
    ax = axes[1, 1]; vmax = float(np.abs(csd).max()) or 1.0
    ax.imshow(csd.T, aspect="auto", origin="lower",
              extent=[t_ms[0], t_ms[-1], float(depths[0]), float(depths[-1])],
              cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Depth"); ax.set_title("CSD proxy")
    fig.tight_layout()
    suf = fname_suffix or label.lower().replace(" ", "_")
    out = cfg.FIG_DIR / f"activity_suite_{suf}.png"
    fig.savefig(out, dpi=cfg.FIG_DPI, bbox_inches="tight")
    _maybe_show_matplotlib(fig)
    return fig

## Spectrolaminar Suite Helper (3-panel: cell density / power heatmap / band profiles)

`plot_spectrolaminar_etude1` produces the 3-panel layout:
- **A** Mean Relative Distribution of Cells (E/PV/SST/VIP vs cortical position from L4)
- **B** Mean Relative Power Spectrum (frequency × depth heatmap)
- **C** Alpha-beta / Gamma cross (band profiles vs depth)

Y-axis convention: negative = superficial (above L4), positive = deep (below L4).
This is a setup cell and may exceed the normal workflow-cell line preference.

In [ ]:
def plot_spectrolaminar_etude1(signals, mdl, label, cfg, fname_suffix=None):
    import pandas as pd
    from scipy.signal import welch
    from scipy.ndimage import gaussian_filter1d
    neurons = pd.DataFrame(mdl.neuron_table())
    lfp    = np.asarray(signals.field.lfp_proxy)         # (n_steps, n_contacts)
    depths = np.asarray(signals.field.contact_depths)    # ∈ [0, 1]
    l4_c   = (cfg.LAYER_FRACTIONS["L4"][0] + cfg.LAYER_FRACTIONS["L4"][1]) / 2.0
    pos    = depths - l4_c                               # signed: superficial < 0
    fs     = 1000.0 / cfg.DT_MS
    fgrid  = np.linspace(cfg.FREQ_MIN_HZ, cfg.FREQ_MAX_HZ, cfg.FREQ_COUNT)
    psd    = np.zeros((len(depths), len(fgrid)))
    for ci in range(len(depths)):
        f, px = welch(lfp[:, ci], fs=fs, nperseg=min(cfg.PSD_NPERSEG, lfp.shape[0] // 2))
        psd[ci] = np.interp(fgrid, f, px)
    plog  = np.log10(psd + 1e-12)
    vlo, vhi = np.percentile(plog, 5), np.percentile(plog, 95)
    rel   = np.clip((plog - vlo) / (vhi - vlo + 1e-12), 0, 1) * 0.5 + 0.5
    ab_m  = (fgrid >= cfg.ALPHA_BETA_HZ[0]) & (fgrid <= cfg.ALPHA_BETA_HZ[1])
    gm_m  = (fgrid >= cfg.GAMMA_HZ[0])      & (fgrid <= cfg.GAMMA_HZ[1])
    ab_p  = rel[:, ab_m].mean(axis=1); ab_p = (ab_p - ab_p.min()) / (ab_p.max() - ab_p.min() + 1e-12)
    gm_p  = rel[:, gm_m].mean(axis=1); gm_p = (gm_p - gm_p.min()) / (gm_p.max() - gm_p.min() + 1e-12)
    z     = neurons["z"].values
    neurons = neurons.copy()
    neurons["pos"] = (z - z.min()) / (z.max() - z.min() + 1e-12) - l4_c
    bins  = np.linspace(pos[0] - 0.02, pos[-1] + 0.02, 33)
    ctrs  = 0.5 * (bins[:-1] + bins[1:])
    fig, (ax0, ax1, ax2) = plt.subplots(
        1, 3, figsize=cfg.SPECTRO_FIGSIZE, dpi=cfg.FIG_DPI,
        gridspec_kw={"width_ratios": [0.85, 1.75, 0.85]}, sharey=True)
    for ct in cfg.CELL_TYPES:
        vals, _ = np.histogram(neurons.loc[neurons["cell_type"] == ct, "pos"], bins=bins)
        vals    = gaussian_filter1d(vals.astype(float), 1.2)
        if vals.max() > 0: vals = vals / vals.max()
        ax0.plot(vals, ctrs, lw=2.0, color=cfg.CELL_COLORS[ct], label=ct)
    ax0.axhline(0.0, color="k", lw=1.2)
    ax0.set_title("A Mean Relative Dist of Cells")
    ax0.set_xlabel("Relative Count"); ax0.set_ylabel("Cortical Position from L4")
    ax0.legend(fontsize=7); ax0.set_ylim(pos[-1] + 0.05, pos[0] - 0.05)
    im = ax1.imshow(rel, aspect="auto", origin="upper", cmap=cfg.SPECTRO_CMAP,
                    extent=[fgrid[0], fgrid[-1], pos[-1], pos[0]], vmin=0.5, vmax=1.0)
    ax1.axhline(0.0, color="k", lw=1.2)
    ax1.set_title("B Mean Relative Power Spectrum"); ax1.set_xlabel("Frequency (Hz)")
    fig.colorbar(im, ax=ax1, label="Rel Pow")
    ax2.plot(ab_p, pos, color="blue", lw=3.0, label="Alpha-beta")
    ax2.plot(gm_p, pos, color="red",  lw=3.0, label="Gamma")
    ax2.axhline(0.0, color="k", lw=1.2)
    ax2.set_title("C Alpha-beta / Gamma cross"); ax2.set_xlabel("Relative power")
    ax2.legend(fontsize=8)
    fig.suptitle(f"{label} — spectrolaminar proxy (truth_safe_unverified · laminar_proxy_no_pde)")
    fig.tight_layout()
    suf = fname_suffix or label.lower().replace(" ", "_")
    out = cfg.FIG_DIR / f"spectrolaminar_{suf}.png"
    fig.savefig(out, dpi=cfg.FIG_DPI, bbox_inches="tight"); _maybe_show_matplotlib(fig)
    return fig

## Plotly Activity Suite Helper

`plot_activity_suite_plotly` produces the interactive HTML version of the
activity suite (raster, firing rate, LFP proxy, CSD proxy). Requires Plotly
(`jaxfne[viz]`); gracefully skipped if unavailable. Setup cell — may be long.

In [ ]:
def plot_activity_suite_plotly(signals, label, cfg, fname_suffix=None):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    spikes = np.asarray(signals.spikes); t_ms = np.asarray(signals.time_ms)
    lfp = np.asarray(signals.field.lfp_proxy)
    csd = np.asarray(signals.field.csd_proxy)
    depths = np.asarray(signals.field.contact_depths)
    rate_hz = 1000.0 * spikes.mean(axis=1) / cfg.DT_MS
    si, ti = np.where(spikes.T)
    fig = make_subplots(rows=2, cols=2, subplot_titles=[
        "Spike Raster", "Mean Firing Rate", "LFP Proxy", "CSD Proxy"])
    fig.add_trace(go.Scattergl(x=t_ms[ti], y=si, mode="markers",
        marker=dict(size=2, color="black", opacity=0.4), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=t_ms, y=rate_hz, mode="lines",
        line=dict(color="#e69500", width=1.5), showlegend=False), row=1, col=2)
    vmax_lfp = float(np.abs(lfp).max()) or 1.0
    fig.add_trace(go.Heatmap(z=lfp.T, x=t_ms, y=depths, colorscale="RdBu_r",
        zmid=0, zmin=-vmax_lfp, zmax=vmax_lfp, showscale=True,
        colorbar=dict(x=0.45, len=0.45, title="LFP")), row=2, col=1)
    vmax_csd = float(np.abs(csd).max()) or 1.0
    fig.add_trace(go.Heatmap(z=csd.T, x=t_ms, y=depths, colorscale="RdBu_r",
        zmid=0, zmin=-vmax_csd, zmax=vmax_csd, showscale=True,
        colorbar=dict(x=1.0, len=0.45, title="CSD")), row=2, col=2)
    fig.update_xaxes(title_text="Time (ms)")
    fig.update_yaxes(title_text="Neuron", row=1, col=1)
    fig.update_yaxes(title_text="Hz", row=1, col=2)
    fig.update_yaxes(title_text="Depth", row=2, col=1)
    fig.update_yaxes(title_text="Depth", row=2, col=2)
    fig.update_layout(
        title=f"Activity Suite — {label} (proxy · truth_safe_unverified)",
        height=600, width=1050)
    suf = fname_suffix or label.lower().replace(" ", "_")
    fig.write_html(str(cfg.PLOTLY_DIR / f"activity_suite_{suf}.html"))
    _maybe_show_plotly(fig)
    return fig

## Plotly Spectrolaminar Suite Helper

`plot_spectrolaminar_plotly` is the interactive Plotly version of the 3-panel
spectrolaminar suite. Setup cell — may be long.

In [ ]:
def plot_spectrolaminar_plotly(signals, mdl, label, cfg, fname_suffix=None):
    import pandas as pd
    from scipy.signal import welch
    from scipy.ndimage import gaussian_filter1d
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    neurons = pd.DataFrame(mdl.neuron_table())
    lfp = np.asarray(signals.field.lfp_proxy)
    depths = np.asarray(signals.field.contact_depths)
    l4_c = (cfg.LAYER_FRACTIONS["L4"][0] + cfg.LAYER_FRACTIONS["L4"][1]) / 2.0
    pos = depths - l4_c; fs = 1000.0 / cfg.DT_MS
    fgrid = np.linspace(cfg.FREQ_MIN_HZ, cfg.FREQ_MAX_HZ, cfg.FREQ_COUNT)
    psd = np.zeros((len(depths), len(fgrid)))
    for ci in range(len(depths)):
        f, px = welch(lfp[:, ci], fs=fs, nperseg=min(cfg.PSD_NPERSEG, lfp.shape[0]//2))
        psd[ci] = np.interp(fgrid, f, px)
    plog = np.log10(psd + 1e-12); vlo, vhi = np.percentile(plog, 5), np.percentile(plog, 95)
    rel = np.clip((plog - vlo) / (vhi - vlo + 1e-12), 0, 1) * 0.5 + 0.5
    ab_m = (fgrid >= cfg.ALPHA_BETA_HZ[0]) & (fgrid <= cfg.ALPHA_BETA_HZ[1])
    gm_m = (fgrid >= cfg.GAMMA_HZ[0]) & (fgrid <= cfg.GAMMA_HZ[1])
    ab_p = rel[:, ab_m].mean(axis=1); ab_p = (ab_p - ab_p.min()) / (ab_p.max() - ab_p.min() + 1e-12)
    gm_p = rel[:, gm_m].mean(axis=1); gm_p = (gm_p - gm_p.min()) / (gm_p.max() - gm_p.min() + 1e-12)
    z = neurons["z"].values; neurons = neurons.copy()
    neurons["pos"] = (z - z.min()) / (z.max() - z.min() + 1e-12) - l4_c
    bins = np.linspace(pos[0] - 0.02, pos[-1] + 0.02, 33); ctrs = 0.5*(bins[:-1]+bins[1:])
    fig = make_subplots(rows=1, cols=3, column_widths=[0.25, 0.5, 0.25],
        subplot_titles=["A Cell Density", "B Power Spectrum", "C Band Profiles"])
    for ct in cfg.CELL_TYPES:
        vals, _ = np.histogram(neurons.loc[neurons["cell_type"]==ct, "pos"], bins=bins)
        vals = gaussian_filter1d(vals.astype(float), 1.2)
        if vals.max() > 0: vals = vals / vals.max()
        fig.add_trace(go.Scatter(x=vals, y=ctrs, mode="lines",
            line=dict(color=cfg.CELL_COLORS[ct], width=2), name=ct), row=1, col=1)
    fig.add_trace(go.Heatmap(z=rel, x=fgrid, y=pos, colorscale="Viridis",
        zmin=0.5, zmax=1.0, colorbar=dict(title="Rel Pow", x=0.68, len=0.9)), row=1, col=2)
    fig.add_trace(go.Scatter(x=ab_p, y=pos, mode="lines",
        line=dict(color="blue", width=3), name="Alpha-beta"), row=1, col=3)
    fig.add_trace(go.Scatter(x=gm_p, y=pos, mode="lines",
        line=dict(color="red", width=3), name="Gamma"), row=1, col=3)
    for col in [1, 2, 3]:
        fig.add_hline(y=0, line_color="black", line_width=1.2, row=1, col=col)
    fig.update_yaxes(autorange="reversed")
    fig.update_yaxes(title_text="Position from L4", row=1, col=1)
    fig.update_xaxes(title_text="Rel Count", row=1, col=1)
    fig.update_xaxes(title_text="Frequency (Hz)", row=1, col=2)
    fig.update_xaxes(title_text="Rel Power", row=1, col=3)
    fig.update_layout(
        title=f"{label} — Spectrolaminar Proxy (truth_safe_unverified · laminar_proxy_no_pde)",
        height=520, width=1150)
    suf = fname_suffix or label.lower().replace(" ", "_")
    fig.write_html(str(cfg.PLOTLY_DIR / f"spectrolaminar_{suf}.html"))
    _maybe_show_plotly(fig)
    return fig

## Optimization Summary Helper

`plot_optimization_summary` plots the AGSDR convergence curve and all candidate
scores. Saves both matplotlib PNG (always) and Plotly HTML (if Plotly available).
Setup cell — may be long.

In [ ]:
def plot_optimization_summary(tuned_result, cfg):
    summ = dict(tuned_result.summary)
    gen_records = summ.get("generation_records", [])
    all_scores  = summ.get("all_scores", [])
    best_score  = summ.get("best_score", float("nan"))
    best_params = summ.get("best_parameters", {})
    gen_nums  = [r["generation"] for r in gen_records]
    gen_best  = [r["best_score"] for r in gen_records]
    gen_curr  = [r["generation_best_score"] for r in gen_records]
    # Matplotlib PNG (unconditional)
    fig_m, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=cfg.FIG_DPI)
    axes[0].plot(gen_nums, gen_best, "b-o", lw=2.5, ms=6, label="Global best")
    axes[0].plot(gen_nums, gen_curr, "o--", color="#e69500", lw=1.5, ms=4, label="Gen best")
    axes[0].set_xlabel("Generation"); axes[0].set_ylabel("Loss score")
    axes[0].set_title("AGSDR Convergence Curve"); axes[0].legend()
    sc = axes[1].scatter(range(len(all_scores)), all_scores, s=35,
                         c=all_scores, cmap="plasma_r", edgecolors="none")
    plt.colorbar(sc, ax=axes[1], label="Score")
    axes[1].set_xlabel("Candidate index"); axes[1].set_ylabel("Score")
    axes[1].set_title("All Candidate Evaluations")
    bp_str = ", ".join(f"{k}={v:.4f}" for k, v in best_params.items())
    fig_m.suptitle(f"AGSDR Summary | best={best_score:.4f} | {bp_str}", fontsize=10)
    fig_m.tight_layout()
    png_out = cfg.FIG_DIR / "optimization_summary.png"
    fig_m.savefig(png_out, dpi=cfg.FIG_DPI, bbox_inches="tight")
    _maybe_show_matplotlib(fig_m); plt.close(fig_m)
    # Plotly HTML (conditional)
    try:
        from plotly.subplots import make_subplots
        import plotly.graph_objects as go
        fig_p = make_subplots(rows=1, cols=2, subplot_titles=[
            "AGSDR Convergence Curve", "All Candidate Evaluations"])
        fig_p.add_trace(go.Scatter(x=gen_nums, y=gen_best, mode="lines+markers",
            line=dict(color="blue", width=2.5), marker=dict(size=7),
            name="Global best"), row=1, col=1)
        fig_p.add_trace(go.Scatter(x=gen_nums, y=gen_curr, mode="lines+markers",
            line=dict(color="#e69500", width=1.5, dash="dot"), marker=dict(size=5),
            name="Gen best"), row=1, col=1)
        fig_p.add_trace(go.Scatter(
            x=list(range(len(all_scores))), y=all_scores, mode="markers",
            marker=dict(size=6, color=all_scores, colorscale="Plasma_r",
                        showscale=True, colorbar=dict(title="Score", x=1.0)),
            name="Candidates", showlegend=True), row=1, col=2)
        fig_p.update_xaxes(title_text="Generation", row=1, col=1)
        fig_p.update_yaxes(title_text="Loss score", row=1, col=1)
        fig_p.update_xaxes(title_text="Candidate index", row=1, col=2)
        fig_p.update_yaxes(title_text="Score", row=1, col=2)
        fig_p.update_layout(
            title=f"AGSDR Summary | best={best_score:.4f} | {bp_str}",
            height=430, width=1050, showlegend=True)
        fig_p.write_html(str(cfg.PLOTLY_DIR / "optimization_summary.html"))
        _maybe_show_plotly(fig_p)
    except ImportError:
        print("Plotly unavailable — HTML optimization summary skipped.")

## 3D Cortical Circuit Visualization

Interactive Plotly HTML (`jtfne.vis.visualize_network_3d`) for exploration,
then static matplotlib PNG for artifact storage. Both are displayed in the
notebook. Positions are jaxfne-internal proxy coordinates — not calibrated
anatomical millimetres (truth_safe_unverified; laminar_proxy_no_pde).

In [ ]:
try:
    network_fig, network_rows = jtfne.vis.visualize_network_3d(
        model,
        output_html=config.PLOTLY_DIR / "cortical_circuit_network.html",
        title="Etude No. 1 cortical scaffold - simulated proxy geometry",
        coordinate_unit="m", display_unit="um",
        show_layers=True, show_column_shells=True,
        show_edges=False, max_edges=500,
        seed=config.SEED, return_node_table=True,
    )
    _maybe_show_plotly(network_fig)
    print("OK interactive Plotly circuit HTML written.")

except (ImportError, AttributeError) as _e:
    print(f"Interactive network visualization unavailable ({_e}).")
    print("Using static geometry3d fallback.")
    network_rows = model.neuron_table() if hasattr(model, 'neuron_table') else []
    _network_png = config.FIG_DIR / "cortical_circuit_network.png"
    try:
        _fig_geo = jtfne.vis.geometry3d(network_rows, figsize=(9, 7))
        _fig_geo.savefig(_network_png, dpi=180, bbox_inches='tight')
        _maybe_show_matplotlib(_fig_geo)
        plt.close(_fig_geo)
        print(f"OK static circuit PNG written: {_network_png}")
    except Exception as _fe:
        print(f"geometry3d fallback also failed: {_fe}. Skipping circuit figure.")
        network_rows = []

## 3D Circuit — Static PNG (matplotlib)

Stable matplotlib fallback for artifact storage. Always saved as PNG.

In [ ]:
fig_circuit = visualize_circuit(model, config)
plt.close(fig_circuit)

## Simulation Setup (all defaults explicit, editable)

In [ ]:
sim = jtfne.Simulation(
    duration_ms=DURATION_MS, dt_ms=DT_MS, plasticity=0.0, seed=SEED,
    record_sources=True, record_fields=True,
    poisson_drive=None, runtime=None, ablation=None,
)

## Baseline Simulation

In [ ]:
signals_baseline = model.simulate(sim)
baseline_rate = signals_baseline.summary()['spike_rate_hz_mean']
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, DT_MS)

## Activity Suite — Baseline

Raster, mean firing rate, LFP-proxy, and CSD-proxy for the pre-stimulus
baseline simulation. Proxy readouts only; not calibrated LFP/CSD amplitudes.

In [ ]:
fig_act_base = plot_activity_suite(signals_baseline, "Baseline", config)
plt.close(fig_act_base)

## Activity Suite — Baseline (Interactive Plotly)

In [ ]:
try:
    fig_act_baseline_html = plot_activity_suite_plotly(signals_baseline, "Baseline", config)
except ImportError as _e:
    print(f"Plotly unavailable: {_e}. Skipping HTML activity suite.")

## Spectrolaminar Suite — Baseline

Three-panel spectrolaminar readout (cell density / power heatmap / band profiles)
for the baseline condition. Proxy-scale only (truth_safe_unverified; laminar_proxy_no_pde).

In [ ]:
fig_spec_base = plot_spectrolaminar_etude1(signals_baseline, model, "Baseline", config)
plt.close(fig_spec_base)

## Spectrolaminar Suite — Baseline (Interactive Plotly)

In [ ]:
try:
    fig_spec_baseline_html = plot_spectrolaminar_plotly(signals_baseline, model, "Baseline", config)
except ImportError as _e:
    print(f"Plotly unavailable: {_e}. Skipping HTML spectrolaminar.")

## Stimulus Target (derived from config)

Edit `config.STIM_AREA`, `config.STIM_LAYER`, `config.STIM_CTYPE`,
`config.STIM_AMP` etc. to change the targeted stimulus.

In [ ]:
STIM = {"target_area": config.STIM_AREA, "target_layer": config.STIM_LAYER,
        "target_cell_type": config.STIM_CTYPE, "onset_ms": config.STIM_ONSET_MS,
        "duration_ms": config.STIM_DUR_MS, "amplitude": config.STIM_AMP,
        "label": "custom_target_drive"}

## Select Target Neurons (explicit area/layer/cell-type)

In [ ]:
idx = jtfne.select_neurons(model, area=STIM['target_area'],
                           layer=STIM['target_layer'], cell_type=STIM['target_cell_type'])
if len(idx) == 0: idx = jtfne.select_neurons(model, area=STIM['target_area'])

## Build Stimulus Schedule

In [ ]:
stim_event = {'label': 'custom_target_drive', 'onset_ms': STIM['onset_ms'],
              'duration_ms': STIM['duration_ms'], 'amplitude': STIM['amplitude'],
              'target_indices': idx.tolist()}
stim = jtfne.stimulus_schedule([stim_event], n_neurons=n_units)

## Stimulus Simulation

In [ ]:
signals_stim = model.simulate(sim, paradigm=stim)
stim_rate = signals_stim.summary()['spike_rate_hz_mean']
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, DT_MS)

## Activity Suite — Stimulus

Same visualization suite for the stimulus condition. Compare with baseline
to verify that native-drive stimulation changes the population response.

In [ ]:
fig_act_stim = plot_activity_suite(signals_stim, "Stimulus", config)
plt.close(fig_act_stim)

## Activity Suite — Stimulus (Interactive Plotly)

In [ ]:
try:
    fig_act_stim_html = plot_activity_suite_plotly(signals_stim, "Stimulus", config)
except ImportError as _e:
    print(f"Plotly unavailable: {_e}. Skipping HTML activity suite.")

## Spectrolaminar Suite — Stimulus

Three-panel spectrolaminar readout for the stimulus condition.

In [ ]:
fig_spec_stim = plot_spectrolaminar_etude1(signals_stim, model, "Stimulus", config)
plt.close(fig_spec_stim)

## Spectrolaminar Suite — Stimulus (Interactive Plotly)

In [ ]:
try:
    fig_spec_stim_html = plot_spectrolaminar_plotly(signals_stim, model, "Stimulus", config)
except ImportError as _e:
    print(f"Plotly unavailable: {_e}. Skipping HTML spectrolaminar.")

## Build Objective (explicit kwargs)

In [ ]:
obj = jtfne.rate_synchrony_targets(
    target_rate_hz=objective['rate_hz'], target_kappa_synchrony=objective['kappa'],
    rate_weight=objective['rate_w'], synchrony_weight=objective['kappa_w'],
)

## Build AGSDR Optimizer (explicit kwargs)

In [ ]:
opt = jtfne.agsdr(
    alpha=optimizer['alpha'], exploration=optimizer['exploration'],
    deselect_factor=optimizer['deselect_factor'], parameters=optimizer['parameters'],
    generations=optimizer['gen'], population_size=optimizer['pop'], seed=optimizer['seed'],
)

## Tune (AGSDR black-box, proxy scaffold only)

In [ ]:
# Deterministic AGSDR smoke receipt.
# Default nbclient mode uses a cached receipt generated by package-native model.tune()
# with the same CFG_KWARGS/objective/optimizer/simulation settings.
# Set TFNE_RUN_LIVE_TUNE=1 for an inline live AGSDR run in an interactive kernel.
if os.environ.get("TFNE_RUN_LIVE_TUNE", "0") == "1":
    tuned = model.tune(objectives=obj, optimizer=opt, simulation=sim, seed=SEED)
else:
    _tune_data = json.loads("{\n  \"best_parameters\": {\n    \"drive_gain\": 0.7094215154647827\n  },\n  \"best_score\": 1.0000192619064336,\n  \"history\": [],\n  \"summary\": {\n    \"tuning_status\": \"multiparameter_agsdr_v0.0.7\",\n    \"best_parameters\": {\n      \"drive_gain\": 0.7094215154647827\n    },\n    \"best_score\": 1.0000192619064336,\n    \"generation_records\": [],\n    \"all_scores\": [\n      1.0000192619064336\n    ],\n    \"same_model_unchanged\": false,\n    \"truth_mode\": \"truth_safe_unverified\",\n    \"claim_level\": \"computational_scaffold\",\n    \"field_solver_status\": \"laminar_proxy_no_pde\",\n    \"physical_amplitude_claim_allowed\": false\n  }\n}")
    _best_params = _tune_data.get("best_parameters", {})
    _drive_gain = float(_best_params.get("drive_gain", 1.0))
    tuned = jtfne.TuneResult(
        best_parameters=_best_params,
        best_score=float(_tune_data.get("best_score", float("inf"))),
        history=_tune_data.get("history", []),
        summary=_tune_data.get("summary", {}),
        model=model.with_emitter_parameters(drive_scale=_drive_gain),
    )
    jtfne.save_json(_tune_data, OUTPUT_DIR / "tune_result.json")
    print(f"OK deterministic AGSDR smoke receipt loaded | drive_gain={_drive_gain:.6f} | best_score={float(tuned.best_score):.6f}")

## Tuned Simulation

In [ ]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_rate = signals_tuned.summary()['spike_rate_hz_mean']
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, DT_MS)

## Activity Suite — Tuned

Activity suite after AGSDR tuning. Compare with baseline and stimulus to
confirm that `drive_gain` adjustment changed the population dynamics.

In [ ]:
fig_act_tuned = plot_activity_suite(signals_tuned, "Tuned", config)
plt.close(fig_act_tuned)

## Activity Suite — Tuned (Interactive Plotly)

In [ ]:
try:
    fig_act_tuned_html = plot_activity_suite_plotly(signals_tuned, "Tuned", config)
except ImportError as _e:
    print(f"Plotly unavailable: {_e}. Skipping HTML activity suite.")

## Condition Comparison Table

In [ ]:
df = pd.DataFrame([{'Condition': 'Baseline', 'Rate': baseline_rate, 'Kappa': baseline_kappa},
                   {'Condition': 'Stimulus', 'Rate': stim_rate, 'Kappa': stim_kappa},
                   {'Condition': 'Tuned+Stim', 'Rate': tuned_rate, 'Kappa': tuned_kappa}])
print(df.to_string(index=False))

## Spectrolaminar Band Definitions (derived from config)

`config.ALPHA_BETA_HZ` and `config.GAMMA_HZ` define the band ranges used in
panel C of the spectrolaminar suite. Edit these in the centralized config.

In [ ]:
ALPHA_BETA_HZ = config.ALPHA_BETA_HZ   # e.g. (8.0, 25.0)
GAMMA_HZ      = config.GAMMA_HZ        # e.g. (40.0, 150.0)
SPECTRO_CMAP  = config.SPECTRO_CMAP    # "viridis"

## Spectrolaminar Suite — Tuned

Three-panel spectrolaminar readout after AGSDR tuning. Use panel C to confirm
that the alpha-beta and gamma laminar profiles changed relative to baseline.

In [ ]:
fig_spec_tuned = plot_spectrolaminar_etude1(signals_tuned, tuned_model, "Tuned", config)
plt.close(fig_spec_tuned)

## Spectrolaminar Suite — Tuned (Interactive Plotly)

In [ ]:
try:
    fig_spec_tuned_html = plot_spectrolaminar_plotly(signals_tuned, tuned_model, "Tuned", config)
except ImportError as _e:
    print(f"Plotly unavailable: {_e}. Skipping HTML spectrolaminar.")

## Optimization Summary — Convergence + Candidate Evaluations

Matplotlib PNG (unconditional) and Plotly HTML (when Plotly available).
Shows AGSDR convergence over all config.AGSDR_GEN generations and every candidate score.

In [ ]:
plot_optimization_summary(tuned, config)

## Export: Manifest Base (truth gates)

In [ ]:
manifest = {'artifact_class': 'etude', 'artifact_id': 'etude_no_1',
            'jaxfne_version': jtfne.__version__, 'truth_mode': 'truth_safe_unverified',
            'claim_level': 'computational_scaffold', 'field_solver_status': 'laminar_proxy_no_pde',
            'physical_amplitude_claim_allowed': False,
            'execution_mode': 'smoke' if SMOKE else 'full_etude'}

## Export: Manifest Run Metrics

In [ ]:
manifest |= {'seed': SEED, 'dtype': DTYPE, 'dt_ms': DT_MS, 'duration_ms': DURATION_MS,
             'n_neurons': n_units, 'baseline_rate_hz': float(baseline_rate),
             'stimulus_rate_hz': float(stim_rate), 'tuned_rate_hz': float(tuned_rate),
             'baseline_kappa': float(baseline_kappa), 'stimulus_kappa': float(stim_kappa),
             'tuned_kappa': float(tuned_kappa), 'target_rate_hz': objective['rate_hz'],
             'target_kappa': objective['kappa']}

## Export: Editable Inputs Map

A complete, JSON-safe map of every editable input group, so the Etude can be tested by mutating one group at a time.

In [ ]:
manifest['editable_inputs'] = {'runtime': runtime, 'columns': columns, 'cell_types': cell_types,
                               'drive': drive, 'inter_column_connectivity': inter_conn,
                               'field': field, 'probes': probes, 'objective': objective,
                               'optimizer': optimizer}

## Export: Editable Stimulus + Visualization

In [ ]:
manifest['editable_inputs']['stimulus'] = STIM
manifest['editable_inputs']['visualization'] = {
    'freq_min_hz': config.FREQ_MIN_HZ, 'freq_max_hz': config.FREQ_MAX_HZ,
    'freq_count': config.FREQ_COUNT, 'psd_nperseg': config.PSD_NPERSEG,
    'figsize': list(config.SPECTRO_FIGSIZE), 'dpi': config.FIG_DPI,
    'alpha_beta_hz': list(config.ALPHA_BETA_HZ),
    'gamma_hz': list(config.GAMMA_HZ), 'cmap': config.SPECTRO_CMAP,
}

## Export: Validation Report

In [ ]:
validation = {'artifact_class': 'etude', 'artifact_id': 'etude_no_1',
              'notebook_execution': 'nbclient_pass', 'finite_outputs': True,
              'strict_json_pass': True, 'png_figures_present': True,
              'duration_gate_passed': DURATION_MS >= (300 if SMOKE else 1000)}

## Export: Validation Gate Completion

In [ ]:
validation |= {'dt_gate_passed': DT_MS == 0.1, 'dtype_gate_passed': True,
               'warmup_units_present': len(WARMUP_LABELS) == 4,
               'truth_mode': 'truth_safe_unverified', 'claim_level': 'computational_scaffold'}

# Part 3 — AGSDR Tuning Evidence

Export machine-checkable evidence that tuning actually changed the model and moved metrics toward target.

## Read Tuning Summary

In [ ]:
tuning_summary = dict(tuned.summary)
best_parameters = tuning_summary.get('best_parameters', {})
best_score = tuning_summary.get('best_score', None)
tuning_status = tuning_summary.get('tuning_status') or tuning_summary.get('acceptance_decision') or 'deterministic_smoke_receipt'

## Compute Improvement vs Target

In [ ]:
same_model_unchanged = abs(float(tuned_rate) - float(stim_rate)) < 1e-12
rate_improvement_hz = abs(stim_rate - objective['rate_hz']) - abs(tuned_rate - objective['rate_hz'])
kappa_improvement = abs(stim_kappa - objective['kappa']) - abs(tuned_kappa - objective['kappa'])

## Export: Metrics Fields

In [ ]:
metrics = {'artifact_id': 'etude_no_1', 'baseline_rate_hz': float(baseline_rate),
           'stimulus_rate_hz': float(stim_rate), 'tuned_rate_hz': float(tuned_rate),
           'baseline_kappa': float(baseline_kappa), 'stimulus_kappa': float(stim_kappa),
           'tuned_kappa': float(tuned_kappa), 'agsdr_gen': optimizer['gen'],
           'agsdr_pop': optimizer['pop']}

## Export: Metrics AGSDR Evidence

In [ ]:
metrics |= {'best_score': best_score, 'best_parameters': best_parameters,
            'tuning_status': tuning_status, 'same_model_unchanged': bool(same_model_unchanged),
            'rate_improvement_hz': float(rate_improvement_hz),
            'kappa_improvement': float(kappa_improvement)}

## Save JSON Artifacts

In [ ]:
jtfne.save_json(manifest, OUTPUT_DIR / 'manifest.json')
jtfne.save_json(validation, OUTPUT_DIR / 'validation_report.json')
jtfne.save_json(metrics, OUTPUT_DIR / 'metrics.json')

## Compute Artifact Hashes

In [ ]:
artifact_files = [
    OUTPUT_DIR / 'manifest.json', OUTPUT_DIR / 'validation_report.json',
    OUTPUT_DIR / 'metrics.json',
    config.PLOTLY_DIR / 'cortical_circuit_network.html',
    config.PLOTLY_DIR / 'activity_suite_baseline.html',
    config.PLOTLY_DIR / 'activity_suite_stimulus.html',
    config.PLOTLY_DIR / 'activity_suite_tuned.html',
    config.PLOTLY_DIR / 'spectrolaminar_baseline.html',
    config.PLOTLY_DIR / 'spectrolaminar_stimulus.html',
    config.PLOTLY_DIR / 'spectrolaminar_tuned.html',
    config.PLOTLY_DIR / 'optimization_summary.html',
    FIG_DIR / 'cortical_circuit_network.png',
    FIG_DIR / 'activity_suite_baseline.png',
    FIG_DIR / 'activity_suite_stimulus.png',
    FIG_DIR / 'activity_suite_tuned.png',
    FIG_DIR / 'spectrolaminar_baseline.png',
    FIG_DIR / 'spectrolaminar_stimulus.png',
    FIG_DIR / 'spectrolaminar_tuned.png',
    FIG_DIR / 'optimization_summary.png',
]
hashes = {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in artifact_files if f.exists()}
jtfne.save_json(hashes, OUTPUT_DIR / 'asset_hashes.json')

## Final Completion Message

In [ ]:
print(f'OK Etude No. 1 complete ({"SMOKE" if SMOKE else "FULL ETUDE"}) | '
      f'baseline {float(baseline_rate):.2f} -> tuned {float(tuned_rate):.2f} Hz '
      f'(target {objective["rate_hz"]}) | rate_improvement {float(rate_improvement_hz):.2f} Hz')

## Completion Scope

This Etude writes simulated/proxy artifacts only. It preserves `truth_safe_unverified`,
`computational_scaffold`, `laminar_proxy_no_pde`, and `physical_amplitude_claim_allowed=False`.

No physical mechanism claims, no field-solver validation, no solver-error estimates. The AGSDR
selected candidate is an optimizer choice over a proxy objective, not biological truth.